In [9]:
import pandas as pd
import patsy
import statsmodels.api as sm
import numpy as np

In [10]:
def validate_design_matrix(X):
    import sympy

    M = sympy.Matrix(X.astype(int).tolist())
    null_vecs = M.nullspace()
    rank = X.shape[1] - len(null_vecs)
    if rank == X.shape[1]:
        return True
    col_names = X.design_info.column_names
    print(f"Design matrix rank: {rank}, shape: {X.shape}")
    print("Linear dependencies:")
    for vec in null_vecs:
        terms = []
        for j, coef in enumerate(vec):
            if coef != 0:
                col = col_names[j]
                if coef.is_Integer:
                    coef = int(coef)
                    if coef == 1:
                        terms.append(col)
                    elif coef == -1:
                        terms.append(f"-{col}")
                    else:
                        terms.append(f"{coef}*{col}")
                else:
                    terms.append(f"{coef}*{col}")
        print(f"  {' + '.join(terms).replace('+ -', '- ')} = 0")
    return False

In [11]:
data = pd.DataFrame(
    {
        "student_left": ["A", "B", "A", "C", "B", "C", "A", "B", "A", "C", "B", "C"],
        "student_right": ["B", "A", "C", "A", "C", "B", "B", "A", "C", "A", "C", "B"],
        "subject": [
            "Math",
            "Physics",
            "Math",
            "Math",
            "Math",
            "Math",
            "Physics",
            "Physics",
            "Math",
            "Physics",
            "Physics",
            "Physics",
        ],
        "studied_left": [
            True,
            False,
            False,
            False,
            True,
            False,
            True,
            False,
            False,
            True,
            True,
            True,
        ],
        "studied_right": [
            False,
            False,
            True,
            True,
            False,
            False,
            False,
            True,
            True,
            True,
            True,
            False,
        ],
        "left_beats_right": [1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0],
    }
)

print(data.to_string())

# Entity terms
entities = sorted(
    set(data["student_left"].unique()).union(set(data["student_right"].unique()))
)
reference = entities[-1]
other_entities = entities[:-1]
entity_terms  = [
    f'I((student_left=="{e}").astype(int) - (student_right=="{e}").astype(int) - (student_left=="{reference}").astype(int) + (student_right=="{reference}").astype(int))'
    for e in other_entities
]

# Affiliation terms
subjects = data["subject"].unique()
subject_terms = [
    f'I((subject=="{m}").astype(int) * (studied_left.astype(int) - studied_right.astype(int)))'
    for m in subjects
]

all_terms = entity_terms + subject_terms
formula = f"left_beats_right ~ 1 + C(subject, Sum) + {' + '.join(all_terms)}"

y, X = patsy.dmatrices(formula, data)

print(pd.DataFrame(X, columns=X.design_info.column_names).to_string())

validate_design_matrix(X)

model = sm.OLS(y, X)
result = model.fit()
print(result.summary())

   student_left student_right  subject  studied_left  studied_right  left_beats_right
0             A             B     Math          True          False                 1
1             B             A  Physics         False          False                 1
2             A             C     Math         False           True                 0
3             C             A     Math         False           True                 0
4             B             C     Math          True          False                 1
5             C             B     Math         False          False                 0
6             A             B  Physics          True          False                 0
7             B             A  Physics         False           True                 1
8             A             C     Math         False           True                 1
9             C             A  Physics          True           True                 1
10            B             C  Physics          True  